# Setup Vector Database for Retrieval Evaluation
## Dataset: [YuITC/Vietnamese-Legal-Documents](https://huggingface.co/datasets/YuITC/Vietnamese-Legal-Documents)
This notebook setup a vector database for evalutating between different embedding models in Vietnamese.

### Dataset Structure

The dataset consists of 2 split: `train` and `test`. Below is the structure of the dataset:

| Column | Description |
|---|---|
| `question` | Natural language query (Vietnamese legal question) |
| `context_list` | List of relevant document passages (ground-truth answers) |
| `qid` | Unique query ID |
| `cid` | List of corpus document IDs that are relevant |

### How to initialize a vector database
1. **Create Corpus**: All unique passages from `context_list` across the dataset are pooled into one corpus.
2. **Setup Vector DB**: Create a consistant client on file systems for testing and evaluating purpose.
3. **Store**: Store the corpus using different embedding models.
4. **Verify**: Check the stored corpus by some simple similarity search (not for evaluation yet).

## 1. Setup environment, libraries

Install necessary dependancies and check cuda/mps availability.

In [1]:
%pip install torch torchvision

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install sentence-transformers datasets pandas numpy matplotlib seaborn pyvi tqdm chromadb

Note: you may need to restart the kernel to use updated packages.


Hardware and Sortware runtime environments:

In [3]:
import torch
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    device = "cuda"
else:    
    device = "cpu"
    
    
import sys
sys.path.append('./utils')
sys.path.append('./metrics')
from utils.load_dataset import load_huggingface_dataset
from metrics import MODELS

PyTorch version : 2.11.0+cu130
CUDA available  : True
GPU             : NVIDIA GeForce RTX 3060 Laptop GPU


/home/ntdatwows2003/Documents/vi_embed_eval/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load the Dataset

Load the Dataset for processing, inspect some metadata and some examples of an instance inside. In this evaluation, we use `test` split for faster processing and less burden for hardware.

In [4]:
dataset_name = "YuITC/Vietnamese-Legal-Documents"

df = load_huggingface_dataset(dataset_name)

Loading YuITC/Vietnamese-Legal-Documents ...


DatasetDict({
    train: Dataset({
        features: ['question', 'context_list', 'qid', 'cid'],
        num_rows: 89261
    })
    test: Dataset({
        features: ['question', 'context_list', 'qid', 'cid'],
        num_rows: 29746
    })
})

Using split : 'test'
Rows        : 29,746
Columns     : ['question', 'context_list', 'qid', 'cid']


## 3. Build corpus

Retrieve unique documents from `context_list` for storing inside vector database.

In [6]:
corpus = {}          # list of unique passage texts (our search index)

for r_idx, data in df.iterrows():
    # print(data["context_list"])
    for p_idx , passage in enumerate(data["context_list"]):
        p = passage.strip()
        pid = f"{data['cid'][p_idx]}"
        if p and pid not in corpus:
            corpus[pid] = p

print(f"Total unique corpus passages : {len(corpus):,}")

Total unique corpus passages : 23,907


## 4. Define embedding Model

In [7]:
print(f"Models to evaluate: {len(MODELS)}")
for m in MODELS:
    print(f"  • {m['label']}")

Models to evaluate: 3
  • AITeamVN
  • Paraphrase-mul-MiniLM-L12-v2
  • BGE-M3


There seems to be some problems with `dangvantuan/vietnamese-document-embedding` and `Alibaba-NLP/gte-multilingual-base`. The errors come from index out of bounds of expected dimension. 

Errors tracebacks point to these line in the model cache when trying to embed sentences:
```
    
```

Further investigation is required.

## 5. Setup Vector Database

In [5]:
import chromadb

client = chromadb.PersistentClient(path="./database")

## 6. Store document corpus for different embedding models

Iterate each models to define embedding function for database collection. Add document corpus into the respective collection.

In [8]:
import time
from tqdm import tqdm
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

print("STORING")
for model in MODELS:
    print(f"\n{'='*65}")
    print(f"  {model['label']}  ({model['name']})")
    print(f"{'='*65}")
    
    print("Creating database collections...")
    
    trust_kargs = {'trust_remote_code': True if 'trust_remote_code' in model else False}
    try:
        collection = client.create_collection(
            name=model["label"],
            embedding_function=SentenceTransformerEmbeddingFunction(
                model_name=model["name"],
                device=device,
                **trust_kargs
            )
        )
        print("Collection created successfully!")
    except Exception as e: 
        print(f"Failed to create collection: {e}")
        # collection = client.get_collection(model['label'])
        # print(f"Collection {model['label']} has already existed!")
        continue
    
    success_num = 0
    print("Processing documents...")
    # Add document into database with retries and error handling
    for cid, context in tqdm(corpus.items(), total=len(corpus), desc="Documents processed"):
        for attempt in range(3):  # Try up to 3 times
            try:
                # Operation that might fail
                collection.add(
                    ids=[cid],
                    documents=[context]
                )
                success_num += 1
                break  # Exit the loop on success
            except Exception as e:
                print(f"Attempt {attempt + 1} failed: {e}. Retrying...")
                time.sleep(2)  # Wait before retrying
        else:
            print("Failed to add documents after 3 attempts. Skipping this document.")
            
    print(f"Processing completed! Successfully added {success_num}/{len(corpus)}")
        

STORING

  AITeamVN  (AITeamVN/Vietnamese_Embedding)
Creating database collections...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 966.73it/s] 


Failed to create collection: Collection [AITeamVN] already exists

  Paraphrase-mul-MiniLM-L12-v2  (sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2)
Creating database collections...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1056.46it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Failed to create collection: Collection [Paraphrase-mul-MiniLM-L12-v2] already exists

  BGE-M3  (BAAI/bge-m3)
Creating database collections...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 7672.75it/s]


Collection created successfully!
Processing documents...


Documents processed: 100%|██████████| 23907/23907 [1:01:17<00:00,  6.50it/s]

Processing completed! Successfully added 23907/23907


## 7. Verify database

Check simple semantic search on the newly created collections.

In [ ]:
print("VERIFYING")
for model in MODELS:
    print(f"\n{'='*65}")
    print(f"  {model['label']}  ({model['name']})")
    print(f"{'='*65}")
    collection = client.get_collection(model["label"])
    retrieve_result = collection.query(
        query_texts=["Phó Tổng Giám đốc Ngân hàng Chính sách xã hội được xếp lương theo bảng lương như thế nào?"],
        n_results=10,
    )

    for i, doc in enumerate(retrieve_result["documents"][0][:5]):
        print(f"Document {i + 1}:")
        # print("Name:", retrieve_result["metadatas"][0][i].get("name", "N/A"))
        print("CID:", retrieve_result["ids"][0][i])
        print("Content:", doc)
        # print("Id:", retrieve_result["ids"][0][i])
        # print("Document number:", retrieve_result["metadatas"][0][i].get("numberDoc", "N/A"))
        # print("Fields:", retrieve_result["metadatas"][0][i].get("fields", "N/A"))
        # print("Metadata:", retrieve_result["metadatas"][0][i])
        print("Score:", retrieve_result["distances"][0][i])
        print("-"* 50)  # Separator for readability

VERIFYING

  AITeamVN  (AITeamVN/Vietnamese_Embedding)
Document 1:
CID: 140864
Content: Áp dụng chế độ tiền lương và phụ cấp quy định tại Nghị định số 26/CP ngày 23 tháng 5 năm 1993 của Chính phủ quy định tạm thời chế độ tiền lương mới trong các doanh nghiệp nhà nước đối với cán bộ, viên chức Ngân hàng Chính sách xã hội như sau:
1. Tổng giám đốc, Phó Tổng giám đốc và Kế toán trưởng, xếp lương theo bảng lương chức vụ quản lý doanh nghiệp hạng đặc biệt.
2. Giám đốc, Phó Giám đốc các chi nhánh, xếp lương theo bảng lương chức vụ quản lý doanh nghiệp theo hạng thực tế đạt được của từng chi nhánh.
Bộ Lao động - Thương binh và Xã hội sau khi trao đổi, thống nhất ý kiến với Bộ Tài chính và Ngân hàng Nhà nước Việt Nam, tạm thời vận dụng xếp hạng cho các chi nhánh trong 3 năm đầu hoạt động.
3. Cán bộ, viên chức chuyên môn, nghiệp vụ xếp lương theo bảng lương viên chức chuyên môn, nghiệp vụ thừa hành phục vụ trong doanh nghiệp.
Score: 0.373146653175354
--------------------------------------------

## 8. Manipulate collections (Optional)

In [ ]:
# client.get_collection("AITeamVN").modify(MODELS[0]["label"])

In [ ]:
# # Delete
# for model in MODELS:
#     client.delete_collection(model["label"])